# Notas — Aula 15: Variabilidade e Linha de Produtos de Software

Marco: a família de robôs (`RoboVeloz`/`RoboExplorador`/`RoboBlindado` × três
estratégias) vira uma linha de produtos configurável — um modelo de features nomeia
cada ponto de variação, e um validador recusa combinações inválidas **antes** de
instanciar o robô.


In [1]:
from enum import Enum

LADO_GRADE = 10

class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


class EstrategiaPadrao:
    def mover(self, robo):
        return robo.avancar()


class EstrategiaEsquiva:
    def mover(self, robo):
        tentativas = 0
        while not robo.sensor_frente() and tentativas < 4:
            robo.girar("DIR")
            tentativas += 1
        return robo.avancar()


class EstrategiaZigzag:
    def __init__(self, periodo=2):
        self.periodo = periodo
        self.passos_dados = 0

    def mover(self, robo):
        if self.passos_dados > 0 and self.passos_dados % self.periodo == 0:
            lado = "DIR" if (self.passos_dados // self.periodo) % 2 else "ESQ"
            robo.girar(lado)
        moveu = robo.avancar()
        if moveu:
            self.passos_dados += 1
        return moveu


class Robo:
    LADO_GRADE = 10
    _registro = {}

    def __init_subclass__(cls, categoria="geral", **kwargs):
        super().__init_subclass__(**kwargs)
        Robo._registro[cls.__name__] = cls
        cls.categoria = categoria

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None, estrategia=None):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}
        self.estrategia = estrategia if estrategia is not None else EstrategiaPadrao()

    def sensor_frente(self):
        dx, dy = self.direcao.value
        nx, ny = self.x + dx, self.y + dy
        return (0 <= nx < Robo.LADO_GRADE and 0 <= ny < Robo.LADO_GRADE
                and (nx, ny) not in self.obstaculos)

    def avancar(self):
        if self.sensor_frente():
            dx, dy = self.direcao.value
            self.x += dx
            self.y += dy
            return True
        return False

    def girar(self, lado):
        ordem = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]
        if lado == "ESQ":
            self.direcao = ordem[(ordem.index(self.direcao) + 1) % 4]
        elif lado == "DIR":
            self.direcao = ordem[(ordem.index(self.direcao) - 1) % 4]

    def mover(self):
        return self.estrategia.mover(self)


class RoboVeloz(Robo, categoria="ofensivo"):
    def avancar(self):
        moveu1 = super().avancar()
        moveu2 = super().avancar()
        return moveu1 or moveu2


class RoboExplorador(Robo, categoria="reconhecimento"):
    pass


class RoboBlindado(Robo, categoria="defensivo"):
    pass


def criar_robo(tipo_nome, nome, **kwargs):
    classe = Robo._registro.get(tipo_nome)
    if classe is None:
        raise ValueError(f"tipo desconhecido: {tipo_nome!r}")
    return classe(nome, **kwargs)


FABRICA_ESTRATEGIAS = {
    "padrao": EstrategiaPadrao,
    "esquiva": EstrategiaEsquiva,
    "zigzag": EstrategiaZigzag,
}


## Modelo de features: mandatória, opcional, alternativa

Toda linha de produtos organiza suas variações em três categorias: **mandatória**
(sempre presente — todo `Robo` tem uma estratégia, mesmo que a padrão), **opcional**
(pode ou não estar presente) e **alternativa** (escolhe-se exatamente uma dentre um
grupo — o tipo do robô, a estratégia de navegação). Hoje esse modelo vira código de
verdade, não só uma ideia no quadro.


In [2]:
ESTRATEGIAS_VALIDAS = {"padrao", "esquiva", "zigzag"}
print(sorted(ESTRATEGIAS_VALIDAS))
print(sorted(Robo._registro))


['esquiva', 'padrao', 'zigzag']
['RoboBlindado', 'RoboExplorador', 'RoboVeloz']


### Sua vez

Complete `TIPOS_VALIDOS`: em vez de copiar os nomes das classes à mão, construa o
conjunto **a partir** de `Robo._registro` — assim, uma subclasse nova entra no
modelo sem precisar editar essa linha.

*Dica: `set(Robo._registro)` — iterar um dicionário dá as chaves.*


In [3]:
# TODO: construa TIPOS_VALIDOS a partir de Robo._registro (não copie os nomes à mão)
TIPOS_VALIDOS = set()

print(sorted(TIPOS_VALIDOS))


[]


## `requires`/`excludes`: restrições entre features

Nem toda combinação de variantes válidas isoladamente faz sentido junto. `EXCLUI`
lista o que não pode coexistir; `REQUER` lista o que precisa coexistir.
`RoboBlindado` é pesado — fazer zigue-zague brusco (`EstrategiaZigzag`) é uma péssima
ideia de engenharia, então ele exclui essa estratégia.


In [4]:
EXCLUI = {
    "RoboBlindado": {"zigzag"},
}
print("zigzag" in EXCLUI.get("RoboBlindado", set()))


True


### Sua vez

Complete `REQUER`: `RoboExplorador` existe pra mapear área — andar sempre reto
(`"padrao"`) desperdiça a vantagem dele. Exija que ele use `"zigzag"` ou `"esquiva"`.

*Dica: `REQUER = {"RoboExplorador": {"zigzag", "esquiva"}}`.*


In [5]:
# TODO: RoboExplorador exige "zigzag" ou "esquiva" — nunca "padrao" sozinho
REQUER = {}

print(REQUER.get("RoboExplorador"))


None


## Validador de configuração: recusar antes de instanciar

Uma exceção customizada sinaliza que uma configuração não passa no modelo.
`validar_configuracao` checa em camadas: a feature existe? combina com as
restrições? Só depois disso um robô pode ser criado.


In [6]:
class ConfiguracaoInvalida(Exception):
    pass


def validar_configuracao(tipo_nome, estrategia_nome):
    if tipo_nome not in Robo._registro:
        raise ConfiguracaoInvalida(f"tipo desconhecido: {tipo_nome!r}")
    if estrategia_nome not in ESTRATEGIAS_VALIDAS:
        raise ConfiguracaoInvalida(f"estratégia desconhecida: {estrategia_nome!r}")
    # TODO: levante ConfiguracaoInvalida se estrategia_nome estiver em
    # EXCLUI.get(tipo_nome, set())
    ...
    exigidas = REQUER.get(tipo_nome)
    if exigidas and estrategia_nome not in exigidas:
        raise ConfiguracaoInvalida(f"{tipo_nome} exige uma destas: {sorted(exigidas)}")


try:
    validar_configuracao("RoboBlindado", "zigzag")
except ConfiguracaoInvalida as erro:
    print(f"{type(erro).__name__}: {erro}")


## Fábrica configurada: encaixando o validador

Igual foi feito em sala: `criar_robo_configurado` ganha uma chamada a
`validar_configuracao` **antes** da primeira linha que cria qualquer coisa.


In [7]:
def criar_robo_configurado(tipo_nome, nome, estrategia_nome="padrao", **kwargs):
    validar_configuracao(tipo_nome, estrategia_nome)
    robo = criar_robo(tipo_nome, nome, **kwargs)
    robo.estrategia = FABRICA_ESTRATEGIAS[estrategia_nome]()
    return robo


robo_bom = criar_robo_configurado("RoboExplorador", "Scout", estrategia_nome="zigzag")
print(robo_bom.nome, type(robo_bom.estrategia).__name__)

try:
    robo_ruim = criar_robo_configurado("RoboBlindado", "Tank", estrategia_nome="zigzag")
except ConfiguracaoInvalida as erro:
    print(f"{type(erro).__name__}: {erro}")


Scout EstrategiaZigzag


## Para aprofundar

- Software Product Line / Linha de Produtos de Software — Wikipedia: https://en.wikipedia.org/wiki/Software_product_line
- Modelo de features (feature model) — Wikipedia: https://en.wikipedia.org/wiki/Feature_model
- Feature Toggles (uma técnica real de variabilidade em produção) — Martin Fowler: https://martinfowler.com/articles/feature-toggles.html
- Exceções customizadas em Python — tutorial oficial: https://docs.python.org/3/tutorial/errors.html#user-defined-exceptions
- `dataclasses` para modelar configuração — documentação oficial: https://docs.python.org/3/library/dataclasses.html
